In [1]:
# input
fasta_file = "./tmp/150_idp_mbp.fasta"
pred_file = "./tmp/150_idp_mbp.tsv"
pred_type_file = "./tmp/150_idp_mbp_pred.tsv"
# output
output_json_file = "./tmp/protenix_input.json"
output_with_metal_json_file = "./tmp/protenix_input_with_metal.json"

In [2]:
metal_type_str_to_input = {
"0": ("ion", "ZN"),
"1": ("ion", "CA"),
"2": ("ion", "MG"),
"3": ("ion", "MN"),
"4": ("ion", "FE"),
"5": ("ion", "CU"),
"6": ("ion", "NI"),
"7": ("ion", "CO"),
"8": ("ligand", "CCD_SF4"),
"9": ("ligand", "CCD_FES"),
"10": ("ligand", "CCD_F3S"),
}

In [3]:
import pandas as pd


df_type = pd.read_table(pred_type_file)
posi_to_type = dict()
for _, row in df_type.iterrows():
    nums = row['pred_seq_num'].split(",")
    metal_types = row['metal_type'].split(",")
    num_to_type = dict(zip(nums, metal_types))
    for n, t in num_to_type.items():
        posi_to_type[(row['seq_id'], int(n) - 1)] = t
del df_type

df = pd.read_table(pred_file)
metal_types = []
for _, row in df.iterrows():
    metal_type = [posi_to_type[(row['seq_id'], int(i))] for i in row['site'].split(',')]
    metal_types.append(",".join(metal_type))
df['metal_type'] = metal_types

In [4]:
df

,seq_id,site,plddt,proba,metal_type
0,K5XPP2,"480,483,488","44.74,38.56,37.15","0.9899,0.9815,0.9399","0,0,0"
1,A0A6A3HSH8,"24,27,32,37","49.54,47.88,41.54,46.23","0.9361,0.9607,0.8607,0.938","0,0,0,0"
2,A0A7G1PCE0,"313,316,323","42.13,47.43,42.41","0.9496,0.8916,0.8925","0,0,0"
3,A0A2W2DWW6,"179,183,191,195","35.92,42.82,43.03,38.94","0.9799,0.9845,0.9243,0.9746","0,0,0,0"
4,G1QBZ9,"318,328,331","39.14,28.1,28.95","0.9561,0.8364,0.8627","0,0,0"
...,...,...,...,...,...
145,A0A2H1W2L1,"39,42,91,94","44.79,39.2,42.99,43.53","0.9731,0.923,0.9747,0.9564","0,0,0,0"
146,D7FSF5,"267,275,278","43.01,43.01,47.25","0.9764,0.9369,0.9156","0,0,0"
147,A0A554SP07,"23,45,48","30.34,32.74,36.66","0.9893,0.9933,0.9459","0,0,0"
148,A0A6P6DE34,"70,90,93","38.63,46.07,46.64","0.8778,0.9778,0.9954","0,0,0"


In [5]:
from Bio import SeqIO

id2seq = dict()
for r in SeqIO.parse(fasta_file, "fasta"):
    id2seq[r.id.split("-")[1]] = str(r.seq)

records = []
records_with_metal = []
for _, row in df.iterrows():
    seq = id2seq[row['seq_id']]

    metal2num = dict()
    for m in set(row['metal_type'].split(",")):
        if m in metal2num.keys():
            metal2num[m] += 1
        else:
            metal2num[m] = 1

    metal_dict_list = []
    for m, num in metal2num.items():
        ligand_type, metal_str = metal_type_str_to_input[m]
        if ligand_type == "ion":
            metal_dict_list.append({
                "ion": {
                    "ion": metal_str,
                    "count": num
                }
            })
        elif ligand_type == "ligand":
            metal_dict_list.append({
                "ligand": {
                    "ligand": f"{metal_str}",
                    "count": num
                }
            })
        else:
            raise ValueError

    pro_chain = [
        {
            "proteinChain": {
                "sequence": seq,
                "count": 1,
                "msa": {
                    "precomputed_msa_dir": f"./tmp/msa_protenix/{row['seq_id']}/0",
                    "pairing_db": "uniref100",
                }
            },
        },
    ] 

    records.append({
        "name": row['seq_id'],
        "sequences": pro_chain
    })

    records_with_metal.append({
        "name": row['seq_id'],
        "sequences": pro_chain + metal_dict_list
    })

In [6]:
import json

with open(output_json_file, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

with open(output_with_metal_json_file, "w", encoding="utf-8") as f:
    json.dump(records_with_metal, f, ensure_ascii=False, indent=4)